# DeepBullwhip Tutorial

## Multi-Tier Supply Chain Bullwhip Effect Simulator

This notebook provides a pedagogical walkthrough of the `deepbullwhip` package. We will:

1. **Generate demand** using a calibrated semiconductor demand model
2. **Build a supply chain** with configurable echelons, ordering policies, and cost functions
3. **Run simulations** and inspect results
4. **Visualize** demand, orders, inventory, costs, and bullwhip metrics
5. **Customize** the chain for different scenarios
6. **Visualize network topology** and geographic layout

---

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Ensure reproducibility
SEED = 42

## 1. Demand Generation

The `SemiconductorDemandGenerator` produces an **AR(1) + seasonal + structural shock** time series, calibrated to WSTS semiconductor data.

$$D_t = \mu + \phi (D_{t-1} - \mu) + A\mu\sin\left(\frac{2\pi t}{52}\right) + S\mu\cdot\mathbb{1}_{t \geq t_s} + \varepsilon_t$$

| Parameter | Default | Meaning |
|-----------|---------|----------|
| `mu` | 50.2 | Monthly mean ($B/month), scaled to weekly |
| `phi` | 0.72 | AR(1) autocorrelation |
| `sigma_eps` | 0.08 | Residual CV |
| `seasonal_amp` | 0.06 | Seasonal amplitude (6%) |
| `shock_period` | 104 | Week when demand shock starts |
| `shock_magnitude` | 0.10 | Shock size (10% increase) |

In [ ]:
from deepbullwhip import SemiconductorDemandGenerator

# Create the generator with default parameters
gen = SemiconductorDemandGenerator()

# Generate 156 weeks (3 years) of demand
demand = gen.generate(T=156, seed=SEED)

print(f"Demand shape: {demand.shape}")
print(f"Mean: {demand.mean():.2f}, Std: {demand.std():.2f}")
print(f"Min: {demand.min():.2f}, Max: {demand.max():.2f}")

In [ ]:
from deepbullwhip.diagnostics.plots import plot_demand_trajectory

fig = plot_demand_trajectory(demand, shock_period=104)
plt.show()

## 2. Understanding the Supply Chain Architecture

The package is built around three **swappable** components:

| Component | Base Class | Default Implementation |
|-----------|-----------|------------------------|
| Demand model | `DemandGenerator` | `SemiconductorDemandGenerator` |
| Ordering policy | `OrderingPolicy` | `OrderUpToPolicy` (OUT) |
| Cost function | `CostFunction` | `NewsvendorCost` |

Each echelon in the chain is composed of an ordering policy and a cost function, injected at construction time. This makes it easy to swap components later.

In [ ]:
from deepbullwhip import OrderUpToPolicy, NewsvendorCost, SupplyChainEchelon

# Build a single echelon manually
policy = OrderUpToPolicy(lead_time=2, service_level=0.95)
cost_fn = NewsvendorCost(holding_cost=0.15, backorder_cost=0.60)

echelon = SupplyChainEchelon(
    name="Distributor",
    lead_time=2,
    policy=policy,
    cost_fn=cost_fn,
    initial_inventory=50.0,
)

print(f"Echelon: {echelon.name}")
print(f"Lead time: {echelon.lead_time} weeks")
print(f"z_alpha (service level): {policy.z_alpha:.3f}")
print(f"Holding cost h={cost_fn.h}, Backorder cost b={cost_fn.b}")

### How the Order-Up-To (OUT) Policy Works

Each period, the echelon computes:

$$S = (L+1) \cdot \hat{\mu} + z_{\alpha} \cdot \hat{\sigma} \cdot \sqrt{L+1}$$

$$\text{IP} = \text{on-hand} + \text{pipeline}$$

$$q = \max(0, S - \text{IP})$$

Let's see this in action:

In [ ]:
# Demonstrate a single ordering decision
order = policy.compute_order(
    inventory_position=30.0,
    forecast_mean=10.0,
    forecast_std=2.0,
)

S = (2 + 1) * 10.0 + policy.z_alpha * 2.0 * np.sqrt(3)
print(f"Order-up-to level S = {S:.2f}")
print(f"Inventory position IP = 30.0")
print(f"Order quantity q = max(0, {S:.2f} - 30.0) = {order:.2f}")

### Newsvendor Cost Function

$$C(I) = \begin{cases} h \cdot I & \text{if } I \geq 0 \quad \text{(holding cost)} \\ b \cdot |I| & \text{if } I < 0 \quad \text{(backorder cost)} \end{cases}$$

In [ ]:
# Holding cost example
print(f"Inventory = +10: cost = {cost_fn.compute(10.0):.2f}  (h * 10 = {0.15 * 10:.2f})")

# Backorder cost example
print(f"Inventory =  -5: cost = {cost_fn.compute(-5.0):.2f}  (b * 5 = {0.60 * 5:.2f})")

# Zero inventory
print(f"Inventory =   0: cost = {cost_fn.compute(0.0):.2f}")

## 3. Building and Simulating the Default Supply Chain

The default configuration models a **4-echelon semiconductor supply chain**:

| Echelon | Role | Lead Time | h | b |
|---------|------|-----------|---|---|
| E1 | Distributor / OEM | 2 weeks | 0.15 | 0.60 |
| E2 | Assembly & Test (OSAT) | 4 weeks | 0.12 | 0.50 |
| E3 | Foundry / Fab | 12 weeks | 0.08 | 0.40 |
| E4 | Wafer / Material Supplier | 8 weeks | 0.05 | 0.30 |

In [ ]:
from deepbullwhip import SerialSupplyChain, default_semiconductor_config

# Inspect the default configuration
for cfg in default_semiconductor_config():
    print(f"{cfg.name:12s}  L={cfg.lead_time:2d}  h={cfg.holding_cost:.2f}  b={cfg.backorder_cost:.2f}")

In [ ]:
# Create the supply chain (uses default config)
chain = SerialSupplyChain()

# Prepare forecasts: for this demo, use rolling mean/std of demand
# In practice, these come from ML models
forecasts_mean = np.full_like(demand, demand.mean())
forecasts_std = np.full_like(demand, demand.std())

# Run the simulation
result = chain.simulate(demand, forecasts_mean, forecasts_std)

print(f"Simulation complete: {len(result.echelon_results)} echelons, {len(demand)} periods")

## 4. Inspecting Results

The simulation returns a `SimulationResult` object containing per-echelon results and aggregate metrics.

In [ ]:
# Print summary metrics
print(f"{'Echelon':<15} {'BW Ratio':>10} {'Fill Rate':>10} {'Total Cost':>12}")
print("-" * 50)
for k, er in enumerate(result.echelon_results):
    print(f"E{k+1}: {er.name:<10} {er.bullwhip_ratio:10.3f} {er.fill_rate:10.1%} {er.total_cost:12,.1f}")
print("-" * 50)
print(f"{'Cumulative BW':<15} {result.cumulative_bullwhip:10.3f}")
print(f"{'Total Cost':<15} {'':>10} {'':>10} {result.total_cost:12,.1f}")

In [ ]:
# The to_dict() method gives a flat dictionary for easy DataFrame conversion
import pandas as pd
pd.Series(result.to_dict()).to_frame("Value").round(3)

## 5. Diagnostic Visualizations

All plots are **publication-grade** and return `Figure` objects. Use the `width` parameter to switch between single-column (3.5") and double-column (7.0") journal formats.

### 5.1 Order Quantities Across Echelons

This shows the **bullwhip effect** visually: how order variability amplifies as you move upstream.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_order_quantities

fig = plot_order_quantities(demand, result)
plt.show()

### 5.2 Order Streams Overlay

All echelon order streams overlaid on customer demand — the classic bullwhip visualization.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_order_streams

fig = plot_order_streams(demand, result)
plt.show()

### 5.3 Inventory On-Hand

Inventory levels per echelon with **backorder shading** (orange regions below zero).

In [ ]:
from deepbullwhip.diagnostics.plots import plot_inventory_levels

fig = plot_inventory_levels(result)
plt.show()

### 5.4 Inventory Position (On-Hand + Pipeline)

The **inventory position** is what the ordering policy targets. It includes on-hand stock plus all orders currently in the pipeline.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_inventory_position

fig = plot_inventory_position(demand, result, chain)
plt.show()

### 5.5 Cost Time Series

Per-period costs decomposed into holding (green) and backorder (gold) components.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_cost_timeseries

fig = plot_cost_timeseries(result)
plt.show()

### 5.6 Echelon Detail View

Deep-dive into a single echelon: orders vs demand, inventory, and costs.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_echelon_detail

# Foundry (E3) — longest lead time, most interesting dynamics
fig = plot_echelon_detail(demand, result, echelon_index=2)
plt.show()

### 5.7 Bullwhip Amplification

Cumulative bullwhip ratio on a **log scale** — shows exponential amplification across echelons.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_bullwhip_amplification

echelon_labels = [er.name for er in result.echelon_results]
fig = plot_bullwhip_amplification({"Default (OUT)": result}, echelon_labels=echelon_labels)
plt.show()

### 5.8 Summary Dashboard

In [ ]:
from deepbullwhip.diagnostics.plots import plot_summary_dashboard

fig = plot_summary_dashboard(demand, result)
plt.show()

## 6. Network Topology & Geographic Map

Visualize the supply chain structure as a network diagram or on a geographic map. The example below uses a Saudi petrochemical supply chain involving KFUPM.

In [ ]:
from deepbullwhip.diagnostics.network import (
    kfupm_petrochemical_network,
    plot_network_diagram,
    plot_supply_chain_map,
)

# Load the KFUPM example network
network = kfupm_petrochemical_network()

for i, node in enumerate(network.nodes):
    print(f"E{i+1}: {node.name} ({node.role})")
    print(f"     Location: ({node.lat:.4f}, {node.lon:.4f})")
    print(f"     {node.details}")
    print()

### 6.1 Network Diagram

In [ ]:
# Without simulation metrics
fig = plot_network_diagram(network)
plt.show()

In [ ]:
# With simulation metrics annotated
fig = plot_network_diagram(network, sim_result=result)
plt.show()

In [ ]:
# Vertical layout (for posters or narrow columns)
fig = plot_network_diagram(network, sim_result=result, orientation="vertical")
plt.show()

### 6.2 Geographic Map

In [ ]:
fig = plot_supply_chain_map(network, sim_result=result)
plt.show()

## 7. Customizing the Supply Chain

You can build custom chains using `EchelonConfig` or by directly composing echelons with different policies and cost functions.

### 7.1 Custom Echelon Configuration

In [ ]:
from deepbullwhip import EchelonConfig

# A simpler 2-echelon chain: retailer + manufacturer
custom_configs = [
    EchelonConfig("Retailer", lead_time=1, holding_cost=0.20, backorder_cost=0.80),
    EchelonConfig("Manufacturer", lead_time=6, holding_cost=0.10, backorder_cost=0.40),
]

custom_chain = SerialSupplyChain.from_config(custom_configs)
custom_result = custom_chain.simulate(demand, forecasts_mean, forecasts_std)

print(f"{'Echelon':<15} {'BW':>8} {'FR':>8} {'Cost':>10}")
print("-" * 45)
for k, er in enumerate(custom_result.echelon_results):
    print(f"E{k+1}: {er.name:<10} {er.bullwhip_ratio:8.3f} {er.fill_rate:8.1%} {er.total_cost:10,.1f}")

### 7.2 Comparing Different Service Levels

In [ ]:
from deepbullwhip.diagnostics.plots import plot_bullwhip_amplification, plot_cost_decomposition

service_levels = [0.80, 0.90, 0.95, 0.99]
results_by_sl = {}

for sl in service_levels:
    configs = [
        EchelonConfig("Distributor", lead_time=2, holding_cost=0.15,
                      backorder_cost=0.60, service_level=sl),
        EchelonConfig("OSAT", lead_time=4, holding_cost=0.12,
                      backorder_cost=0.50, service_level=sl),
        EchelonConfig("Foundry", lead_time=12, holding_cost=0.08,
                      backorder_cost=0.40, service_level=sl),
        EchelonConfig("Supplier", lead_time=8, holding_cost=0.05,
                      backorder_cost=0.30, service_level=sl),
    ]
    ch = SerialSupplyChain.from_config(configs)
    results_by_sl[f"SL={sl:.0%}"] = ch.simulate(demand, forecasts_mean, forecasts_std)

fig = plot_bullwhip_amplification(
    results_by_sl,
    echelon_labels=["Distributor", "OSAT", "Foundry", "Supplier"],
)
plt.show()

In [ ]:
fig = plot_cost_decomposition(results_by_sl)
plt.show()

### 7.3 Custom Demand Generator

In [ ]:
# Higher volatility, stronger shock
volatile_gen = SemiconductorDemandGenerator(
    mu=50.2,
    phi=0.85,            # stronger autocorrelation
    sigma_eps=0.15,      # higher noise
    shock_magnitude=0.25 # 25% demand spike
)

volatile_demand = volatile_gen.generate(T=156, seed=SEED)

fig = plot_demand_trajectory(volatile_demand, shock_period=104)
plt.show()

### 7.4 Implementing a Custom Ordering Policy

You can create your own policy by subclassing `OrderingPolicy`. Here's a simple fixed-order-quantity policy:

In [ ]:
from deepbullwhip.policy.base import OrderingPolicy


class FixedOrderPolicy(OrderingPolicy):
    """Order a fixed quantity every period."""

    def __init__(self, fixed_quantity: float) -> None:
        self.fixed_quantity = fixed_quantity

    def compute_order(self, inventory_position, forecast_mean, forecast_std):
        return self.fixed_quantity


# Build a chain with the fixed-order policy
fixed_echelons = []
for cfg in default_semiconductor_config():
    total_h = cfg.holding_cost + cfg.depreciation_rate
    fixed_echelons.append(
        SupplyChainEchelon(
            name=cfg.name,
            lead_time=cfg.lead_time,
            policy=FixedOrderPolicy(demand.mean()),
            cost_fn=NewsvendorCost(total_h, cfg.backorder_cost),
            initial_inventory=cfg.initial_inventory,
        )
    )

fixed_chain = SerialSupplyChain(echelons=fixed_echelons)
fixed_result = fixed_chain.simulate(demand, forecasts_mean, forecasts_std)

print(f"Fixed-order BW_cumulative: {fixed_result.cumulative_bullwhip:.3f}")
print(f"OUT policy BW_cumulative:  {result.cumulative_bullwhip:.3f}")

## 8. Saving Figures for Publication

All plot functions return `Figure` objects that you can save as high-resolution PDFs:

In [ ]:
# Example: save a single-column figure as PDF
fig = plot_bullwhip_amplification(
    {"OUT": result, "Fixed": fixed_result},
    echelon_labels=["Distributor", "OSAT", "Foundry", "Supplier"],
    width="single",
)

# Uncomment to save:
# fig.savefig("bullwhip_comparison.pdf", dpi=300)

plt.show()
print(f"Figure size: {fig.get_size_inches()} inches")

## 9. Metrics Module

Standalone metric functions for use in custom analyses:

In [ ]:
from deepbullwhip.diagnostics.metrics import (
    bullwhip_ratio,
    fill_rate,
    cumulative_bullwhip,
    bullwhip_lower_bound,
)

# Direct computation from arrays
er0 = result.echelon_results[0]
bw = bullwhip_ratio(er0.orders, demand)
fr = fill_rate(er0.inventory_levels)

print(f"E1 Bullwhip ratio: {bw:.3f}")
print(f"E1 Fill rate: {fr:.1%}")

# Theoretical lower bound (Theorem 1)
lb = bullwhip_lower_bound(lead_time=2, sensitivity=0.5, phi=0.72)
print(f"\nTheorem 1 lower bound (L=2, lambda=0.5, phi=0.72): {lb:.3f}")

---

## Summary

The `deepbullwhip` package provides:

- **Modular architecture** with swappable demand generators, ordering policies, and cost functions
- **Configurable multi-echelon chains** via `EchelonConfig` or direct composition
- **Publication-grade diagnostics** with KFUPM AI-VNV Lab color palette
- **Network & geographic visualizations** for supply chain topology
- **Typed results** (`SimulationResult`, `EchelonResult`) with `to_dict()` for easy analysis

Next steps:
- Hook up ML demand forecasters (LSTM, Transformer, etc.)
- Implement additional ordering policies (e.g., (s, S), MMFE)
- Run Monte Carlo experiments to study the bullwhip-accuracy tradeoff